# Bootstrap Robustness Check — EA Smoking Status

**Purpose:** Same method as AA. Given this network already showed the weakest
alpha-sensitivity result (0 genes survive alpha=0.0001) and carries the
unresolved λ=1.66 inflation, this bootstrap check is a further honest stress-test.

**Input:** `pc_input_smoking_sensitivity.npy`, `pc_col_names_smoking_sensitivity.json`.

In [2]:
import numpy as np
import json
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
from collections import defaultdict
import time

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
X_pc_full_smk_ea = np.load(os.path.join(out_dir, "pc_input_smoking_sensitivity.npy"))
with open(os.path.join(out_dir, "pc_col_names_smoking_sensitivity.json")) as f:
    col_names_smk_ea = json.load(f)

n_nodes = len(col_names_smk_ea)
outcome_idx = col_names_smk_ea.index("smoking_status")
n_samples = X_pc_full_smk_ea.shape[0]

def get_direct_parents_ea_smk(X, alpha_val=0.001):
    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in col_names_smk_ea]
    for i in range(n_nodes - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    cg = pc(data=X, alpha=alpha_val, indep_test=fisherz, stable=True,
            uc_rule=0, uc_priority=2, background_knowledge=bk,
            verbose=False, show_progress=False, node_names=col_names_smk_ea)

    adj = cg.G.graph
    direct_parents = set()
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if adj[i,j] == -1 and adj[j,i] == 1 and col_names_smk_ea[j] == "smoking_status":
                direct_parents.add(col_names_smk_ea[i])
            elif adj[i,j] == 1 and adj[j,i] == -1 and col_names_smk_ea[i] == "smoking_status":
                direct_parents.add(col_names_smk_ea[j])
            elif adj[i,j] == -1 and adj[j,i] == -1:
                if col_names_smk_ea[i] == "smoking_status":
                    direct_parents.add(col_names_smk_ea[j])
                elif col_names_smk_ea[j] == "smoking_status":
                    direct_parents.add(col_names_smk_ea[i])
    return direct_parents

n_bootstraps = 100
edge_counts_ea_smk = defaultdict(int)
n_failed = 0
n_succeeded = 0

np.random.seed(0)
start = time.time()
for b in range(n_bootstraps):
    boot_idx = np.random.choice(n_samples, n_samples, replace=True)
    X_boot = X_pc_full_smk_ea[boot_idx]
    try:
        parents = get_direct_parents_ea_smk(X_boot)
        for gene in parents:
            edge_counts_ea_smk[gene] += 1
        n_succeeded += 1
    except (ValueError, np.linalg.LinAlgError) as e:
        n_failed += 1
        continue
    if (b + 1) % 20 == 0:
        print(f"Completed {b+1}/{n_bootstraps}, elapsed {time.time()-start:.1f}s")

print(f"\nTotal time: {time.time()-start:.1f}s")
print(f"Succeeded: {n_succeeded}, Failed (singular matrix): {n_failed}")
print(f"\nEdge stability across {n_succeeded} successful bootstraps:")
for gene, count in sorted(edge_counts_ea_smk.items(), key=lambda x: -x[1]):
    pct = count / n_succeeded * 100
    print(f"  {gene}: {count}/{n_succeeded} ({pct:.0f}%)")

Completed 20/100, elapsed 1.7s
Completed 40/100, elapsed 3.2s
Completed 80/100, elapsed 6.0s
Completed 100/100, elapsed 7.5s

Total time: 7.5s
Succeeded: 93, Failed (singular matrix): 7

Edge stability across 93 successful bootstraps:
  exm935491-0_T_R_1918372056: 51/93 (55%)
  exm1387779-0_B_F_1921595005: 49/93 (53%)
  exm793377-0_T_R_1922407882: 49/93 (53%)
  exm71047-0_B_R_1921357564: 48/93 (52%)
  exm421337-0_B_F_1923050859: 45/93 (48%)
  exm619924-0_B_F_1918533121: 39/93 (42%)
  exm2251364-0_T_F_1975257170: 38/93 (41%)
  exm1441545-0_B_R_2058868341: 28/93 (30%)
  exm1413456-0_B_F_1923224907: 18/93 (19%)
  exm1242904-0_T_F_1921733541: 8/93 (9%)
  exm695084-0_B_R_1922987914: 7/93 (8%)
